In [ ]:
!pip install lightning

In [ ]:
!unzip trafic_32.zip -d /data/

Streaming output truncated to the last 5000 lines.
   creating: /data/trafic_32/00014/00000_00009/
  inflating: /data/trafic_32/00014/00000_00009/00000_00009.jpg  
   creating: /data/trafic_32/00014/00010_00017/
  inflating: /data/trafic_32/00014/00010_00017/00010_00017.jpg  
   creating: /data/trafic_32/00014/00012_00001/
  inflating: /data/trafic_32/00014/00012_00001/00012_00001.jpg  
   creating: /data/trafic_32/00014/00021_00028/
  inflating: /data/trafic_32/00014/00021_00028/00021_00028.jpg  
   creating: /data/trafic_32/00014/00001_00004/
  inflating: /data/trafic_32/00014/00001_00004/00001_00004.jpg  
   creating: /data/trafic_32/00014/00018_00022/
  inflating: /data/trafic_32/00014/00018_00022/00018_00022.jpg  
   creating: /data/trafic_32/00014/00002_00001/
  inflating: /data/trafic_32/00014/00002_00001/00002_00001.jpg  
   creating: /data/trafic_32/00014/00020_00016/
  inflating: /data/trafic_32/00014/00020_00016/00020_00016.jpg  
   creating: /data/trafic_32/00014/00002_0001

In [ ]:
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, random_split
from collections import Counter


class RoadSignsDataset(Dataset):
    def __init__(self, subset, transform) -> None:
        super().__init__()
        self.dataset = subset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        x, y = self.dataset[index]
        if self.transform:
            x = self.transform(x)
        return x, y


def load_data(train_transform, val_transform, batch_size=64, seed=42):
    dataset = ImageFolder("/data/trafic_32/")

    class_counts = Counter(dataset.targets)
    total_samples = sum(class_counts.values())

    train_size = int(0.8 * total_samples)
    val_size = total_samples - train_size
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(dataset, [train_size, val_size], generator=generator)

    train_dataset = RoadSignsDataset(train_subset, train_transform)
    val_dataset = RoadSignsDataset(val_subset, val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader, dataset.class_to_idx

In [ ]:
import lightning as L
from lightning.pytorch.loggers import WandbLogger
import torch
import torch.nn as nn
import torchvision
import wandb


class LitVAE(L.LightningModule):
    def __init__(self, vae_model: nn.Module, num_epochs) -> None:
        super().__init__()
        self.num_epochs = num_epochs
        self.model = vae_model
        self.reconstruction_criterion = nn.MSELoss(reduction='sum')

    def training_step(self, batch, batch_idx):
        x, _ = batch
        reconstructed_image, mu, var_log = self.model(x)

        recon_loss = self.reconstruction_criterion(reconstructed_image, x) / x.size(0)
        kl_loss = self._calculate_kl_loss(mu, var_log)
        loss = recon_loss + kl_loss

        self.log("train/recon_loss", recon_loss, prog_bar=False)
        self.log("train/kl_loss", kl_loss, prog_bar=False)
        self.log("train/loss", loss, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        reconstructed_image, mu, var_log = self.model(x)

        recon_loss = self.reconstruction_criterion(reconstructed_image, x) / x.size(0)
        kl_loss = self._calculate_kl_loss(mu, var_log)
        loss = recon_loss + kl_loss

        self.log("val/loss", loss, prog_bar=True)

        if batch_idx == 0 and isinstance(self.logger, WandbLogger):
            n = min(x.size(0), 8)
            comparison = torch.cat([x[:n], reconstructed_image[:n]])
            grid = torchvision.utils.make_grid(comparison, nrow=n, normalize=False)
            self.logger.experiment.log({"val/reconstructions": [wandb.Image(grid)]})

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-5)
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.num_epochs, eta_min=1e-5)
        return [optimizer], [lr_scheduler]

    def _calculate_kl_loss(self, mu, var_log):
        kl_loss = -0.5 * torch.sum(1 + var_log - mu.pow(2) - var_log.exp(), dim=[1, 2, 3])
        return kl_loss.mean()


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch) -> None:
        super().__init__()
        self.relu = nn.ReLU()

        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

        self.bnorm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.bnorm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

    def forward(self, x):
        out = self.bnorm1(x)
        out = self.relu(out)
        out = self.conv1(out)

        out = self.bnorm2(out)
        out = self.relu(out)
        out = self.conv2(out)

        return out + self.res_conv(x)


class VAEModel(nn.Module):
    def __init__(self, image_channels=3, block_out_channels=(32, 64, 128)) -> None:
        super().__init__()

        self.initial_conv = nn.Conv2d(image_channels, block_out_channels[0], 3, padding=1)

        # ENCODER
        self.encoder_blocks = nn.ModuleList()
        self.downsample_blocks = nn.ModuleList()
        for i in range(len(block_out_channels)-1):
            res_block = ResBlock(block_out_channels[i], block_out_channels[i])
            downsample_block = nn.Conv2d(block_out_channels[i], block_out_channels[i+1], 4, 2, padding=1)

            self.encoder_blocks.append(res_block)
            self.downsample_blocks.append(downsample_block)

        self.encoder_output = nn.Conv2d(block_out_channels[-1], 2*block_out_channels[-1], 3, padding=1) # mu and log(var)

        # DECODER
        self.decoder_blocks = nn.ModuleList()
        self.upsample_blocks = nn.ModuleList()
        for i in range(1, len(block_out_channels)):
            upsample_block = nn.ConvTranspose2d(block_out_channels[-i], block_out_channels[-(i+1)], 4, 2, padding=1)
            res_block = ResBlock(block_out_channels[-(i+1)], block_out_channels[-(i+1)])

            self.decoder_blocks.append(res_block)
            self.upsample_blocks.append(upsample_block)

        self.decoder_output = nn.Conv2d(block_out_channels[0], image_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def reparameterize(self, mu, var_log):
        std = torch.exp(0.5 * var_log)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        out = z
        for conv_block, upsample_block in zip(self.decoder_blocks, self.upsample_blocks):
            out = conv_block(upsample_block(out))

        recostructed_image = self.decoder_output(out)
        return self.sigmoid(recostructed_image)

    def forward(self, x):
        out = self.initial_conv(x)

        for conv_block, downsample_block in zip(self.encoder_blocks, self.downsample_blocks):
            out = downsample_block(conv_block(out))

        encoder_output = self.encoder_output(out)
        mu, var_log = torch.chunk(encoder_output, 2, dim=1)

        z = self.reparameterize(mu, var_log)    # sample from latent space

        reconstructed_scaled_image = self.decode(z)
        return reconstructed_scaled_image, mu, var_log

In [ ]:
import wandb
import torch
import torchvision
import lightning as L
from lightning.pytorch.loggers import WandbLogger
import torch.nn as nn
import math


class LiDiffusion(L.LightningModule):
    def __init__(self, diffusion_model: nn.Module, num_epochs, diffusion_steps=1000) -> None:
        super().__init__()

        self.model = diffusion_model
        self.criterion = nn.MSELoss()

        self.num_epochs = num_epochs
        self.diffusion_steps = diffusion_steps

        self.register_buffer("betas", torch.linspace(1e-4, 0.02, diffusion_steps))
        self.register_buffer("alphas", 1. - self.betas) # type: ignore
        self.register_buffer("alphas_cumprod", torch.cumprod(self.alphas, dim=-1)) # type: ignore

    def _calculate_diffusion_loss(self, x, device):
        timestamps = torch.randint(0, self.diffusion_steps, size=(x.shape[0],), device=device)
        alphas_cum = self.alphas_cumprod[timestamps].view(-1, 1, 1, 1) # type: ignore

        noise = torch.randn_like(x)
        noisy_x = torch.sqrt(alphas_cum) * x + torch.sqrt(1. - alphas_cum) * noise

        predicted_noise = self.model(noisy_x, timestamps)
        loss = self.criterion(predicted_noise, noise)
        return loss

    def training_step(self, batch, batch_idx):
        x, _ = batch
        device = x.device

        loss = self._calculate_diffusion_loss(x, device)

        self.log("train/loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        device = x.device

        loss = self._calculate_diffusion_loss(x, device)
        self.log("val/loss", loss, prog_bar=True)

        if batch_idx == 0 and isinstance(self.logger, WandbLogger):
            n = min(x.size(0), 16)

            reconstructed_images = self.generate_batch(n, device)

            grid = torchvision.utils.make_grid(
                reconstructed_images, nrow=4, normalize=True, value_range=(-1, 1)
            )
            self.logger.experiment.log({"val/reconstructions": [wandb.Image(grid)]})

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-5)
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.num_epochs, eta_min=1e-5)
        return [optimizer], [lr_scheduler]

    def generate_batch(self, batch_size, device):
        reconstructed_images = torch.randn((batch_size, 3, 32, 32), device=device)

        for t in reversed(range(self.diffusion_steps)):
            timestamps = torch.full((batch_size,), t, device=device, dtype=torch.long)

            with torch.no_grad():
                # self.model to teraz referencja do U-Netu wewnątrz LiDiffusion
                predicted_noise = self.model(reconstructed_images, timestamps)

            alpha_t = self.alphas[t]                # type: ignore
            alpha_cum_t = self.alphas_cumprod[t]    # type: ignore
            beta_t = self.betas[t]                  # type: ignore

            coef1 = 1 / torch.sqrt(alpha_t)
            coef2 = (1 - alpha_t) / torch.sqrt(1 - alpha_cum_t)

            mu = coef1 * (reconstructed_images - coef2 * predicted_noise)

            if t > 0:
                z = torch.randn_like(reconstructed_images)
                sigma = torch.sqrt(beta_t)
                reconstructed_images = mu + sigma * z
            else:
                reconstructed_images = mu

        return torch.clamp(reconstructed_images, -1.0, 1.0)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.size = size

    def forward(self, timestamp_indxes):
        device = timestamp_indxes.device

        half_dim = self.size // 2
        embeddings = torch.exp(-1 * torch.arange(half_dim, device=device) * math.log(10000)/(half_dim-1))
        embeddings = timestamp_indxes[:, None] * embeddings[None, :]

        return torch.concatenate((torch.sin(embeddings), torch.cos(embeddings)), dim=-1)


class ResBlockDiffusion(nn.Module):
    def __init__(self, in_ch, out_ch, timestamp_emb_dim) -> None:
        super().__init__()
        self.act = nn.SiLU()
        self.time_mlp = nn.Linear(timestamp_emb_dim, out_ch)
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

        self.bnorm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)

        self.bnorm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)

    def forward(self, x, timestamps):
        out = self.bnorm1(x)
        out = self.act(out)
        out = self.conv1(out)

        timestamps = self.time_mlp(self.act(timestamps))
        timestamps_matrix = timestamps[:, :, None, None]
        out = out + timestamps_matrix

        out = self.bnorm2(out)
        out = self.act(out)
        out = self.conv2(out)

        return out + self.res_conv(x)


class DiffusionModel(nn.Module):
    def __init__(self, image_channels=3, block_out_channels=(32, 64, 128), time_emb_dim=128, diffusion_steps=1000):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )

        self.initial_conv = nn.Conv2d(image_channels, block_out_channels[0], kernel_size=3, padding=1)

        # Unet ENCODER
        self.encoder_blocks = nn.ModuleList()
        self.downsample_blocks = nn.ModuleList()
        for i in range(len(block_out_channels)-1):
            res_block = ResBlockDiffusion(block_out_channels[i], block_out_channels[i], time_emb_dim)
            downsample_block = nn.Conv2d(block_out_channels[i], block_out_channels[i+1], 4, 2, padding=1)

            self.encoder_blocks.append(res_block)
            self.downsample_blocks.append(downsample_block)

        # Unet Bottleneck
        self.bottleneck_layer = ResBlockDiffusion(block_out_channels[-1], block_out_channels[-1], time_emb_dim)

        # Unet DECODER
        self.decoder_blocks = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        for i in range(1, len(block_out_channels)):
            unsample_block = nn.ConvTranspose2d(block_out_channels[-i], block_out_channels[-(i+1)], 4, 2, padding=1)
            res_block = ResBlockDiffusion(2 * block_out_channels[-(i+1)], block_out_channels[-(i+1)], time_emb_dim)

            self.decoder_blocks.append(res_block)
            self.upsamples.append(unsample_block)

        self.output_conv = nn.Conv2d(block_out_channels[0], image_channels, 1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        out = self.initial_conv(x)

        residuals = []

        for conv_block, downsample_block in zip(self.encoder_blocks, self.downsample_blocks):
            out = conv_block(out, t_emb)
            residuals.append(out)
            out = downsample_block(out)

        out = self.bottleneck_layer(out, t_emb)

        for conv_block, upsample_block in zip(self.decoder_blocks, self.upsamples):
            out = upsample_block(out)
            residual_x = residuals.pop()
            out = conv_block(torch.concat([out, residual_x], dim=1), t_emb)

        out = self.output_conv(out)
        return out


In [ ]:
import torchvision.transforms as transforms


shared_train_transforms = [
    transforms.RandomCrop(32, padding=2, padding_mode='edge'),
    transforms.ToTensor(),
]
shared_test_transforms = [
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
]

vae_config = {
    "name": "vae",
    "model_class": VAEModel,
    "lightning_model_class": LitVAE,
    "latent_dim": (128, 8, 8),
    "model_args": {
        "image_channels": 3,
        "block_out_channels": (32, 64, 128)
    },
    "lightning_model_class_args": {
        "num_epochs": 500,
    },
    "train_transform": transforms.Compose(shared_train_transforms),
    "test_transform": transforms.Compose(shared_test_transforms)
}

diffusion_config = {
    "name": "diffusion",
    "model_class": DiffusionModel,
    "lightning_model_class": LiDiffusion,
    "latent_dim": (128, 8, 8),
    "model_args": {
        "image_channels": 3,
        "block_out_channels": (32, 64, 128),
        "time_emb_dim": 128,
        "diffusion_steps": 1000
    },
    "lightning_model_class_args": {
        "num_epochs": 500,
        "diffusion_steps": 1000
    },
    "train_transform": transforms.Compose([
        *shared_train_transforms,
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ]),
    "test_transform": transforms.Compose([
        *shared_test_transforms,
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
}


In [ ]:
import wandb
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint


def train(config):
    train_transform = config["train_transform"]
    test_transform = config["test_transform"]

    train_loader, val_loader, class_to_idx = load_data(train_transform, test_transform, batch_size=64)
    num_classes=len(class_to_idx.keys())

    raw_model = config["model_class"](**config["model_args"])
    model = config["lightning_model_class"](raw_model, **config["lightning_model_class_args"])

    checkpoint_callback = ModelCheckpoint(
        dirpath=f"checkpoints/{config['name']}",
        filename="ckp-{epoch:02d}-{val_loss:.4f}",
        monitor="val/loss",
        mode="min",
        save_top_k=1,
        save_last=True
    )
    wandb_logger = WandbLogger(project="MyExperiments")
    trainer = L.Trainer(
        max_epochs=config["lightning_model_class_args"]["num_epochs"],
        accelerator='gpu',
        devices=1,
        precision="16-mixed",   # Faster training
        logger=wandb_logger,
        callbacks=[checkpoint_callback]
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    wandb.finish()

In [ ]:
import os
from torchvision.utils import save_image
import torch


def generate_samples(config, checkpoint_path: str, num_samples=1000, batch_size=100):
    output_dir=f"generated_samples/{config['name']}"
    os.makedirs(output_dir, exist_ok=True)

    raw_model = config["model_class"](**config["model_args"])

    kwargs = {"vae_model": raw_model} if config["name"] == "vae" else {"diffusion_model": raw_model}
    kwargs.update(config["lightning_model_class_args"])
    lit_model = config["lightning_model_class"].load_from_checkpoint(
        checkpoint_path,
        **kwargs
    )
    model = lit_model.model   # type: ignore

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    generated_count = 0
    with torch.no_grad():

        while generated_count < num_samples:
            current_batch_size = min(batch_size, num_samples - generated_count)

            if config["name"] == "vae":
                latent_dim = config["latent_dim"]

                z = torch.randn(current_batch_size, *latent_dim).to(device)
                generated_images = model.decode(z)

                for i in range(current_batch_size):
                    filename = os.path.join(output_dir, f"sample_{generated_count + i:04d}.png")
                    save_image(generated_images[i], filename)

                generated_count += current_batch_size
                print(f"Generation for ({config['name']}): {generated_count}/{num_samples}")

            elif config["name"] == "diffusion":
                generated_images = lit_model.generate_batch(
                    batch_size=current_batch_size,
                    device=device
                )
                for i in range(current_batch_size):
                    filename = os.path.join(output_dir, f"sample_{generated_count + i:04d}.png")
                    save_image(
                        generated_images[i],
                        filename,
                        normalize=True,
                        value_range=(-1.0, 1.0)
                    )

                generated_count += current_batch_size
                print(f"Generation for ({config['name']}): {generated_count}/{num_samples}")

    print(f"Generated and saved {num_samples} images in directory: {output_dir}")


In [ ]:
train(diffusion_config)

Epoch 171/499 ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/491 0:00:01 • 0:01:03 7.69it/s v_num: tvd6 train/loss: 0.024     
                                                                                 val/loss: 0.024                   